# DentaBridge: tuning Qwen3-VL-4B on one free T4

Settings: **Accelerator** = GPU T4 (x2 is fine, one GPU is used), **Internet** = On. Add the private
dataset `dentabridge-code` (built with `make kaggle-bundle`) as an input.

Run the cells in order. The smoke cell prints the time the full run will take: stop there if that is
more than you want to spend (a session lasts at most 12 hours and the weekly GPU quota is limited).
The same stages run on Colab with `WORK=/content SCRATCH=/content/scratch ROOT=/content/dentabridge`.

In [ ]:
import pathlib
import subprocess

ROOT = pathlib.Path("/kaggle/working/dentabridge")
if not ROOT.exists():
    bundle = next(pathlib.Path("/kaggle/input").rglob("dentabridge-code.tar.gz"))
    subprocess.run(["tar", "-xzf", str(bundle), "-C", "/kaggle/working"], check=True)
print("code at commit", (ROOT / "COMMIT").read_text().strip())

## 1. Install

In [ ]:
!bash /kaggle/working/dentabridge/scripts/kaggle/run.sh setup

## 2. Data (AKU and AlphaDent from Zenodo, CC BY 4.0)

In [ ]:
!bash /kaggle/working/dentabridge/scripts/kaggle/run.sh data

## 3. Smoke test: ten steps, and the estimate for the full run

In [ ]:
!bash /kaggle/working/dentabridge/scripts/kaggle/run.sh smoke

## 4. Train (resumes from the newest checkpoint if the session was interrupted)

In [ ]:
!bash /kaggle/working/dentabridge/scripts/kaggle/run.sh train

## 5. Merge the adapter and score the validation split

In [ ]:
!bash /kaggle/working/dentabridge/scripts/kaggle/run.sh merge

In [ ]:
!bash /kaggle/working/dentabridge/scripts/kaggle/run.sh eval

## 6. Baseline: the same prompt on the untuned model

In [ ]:
!bash /kaggle/working/dentabridge/scripts/kaggle/run.sh zeroshot

## 7. Results

In [ ]:
!cat /kaggle/working/out/*/val/val.md

Download from the **Output** tab: `out/lean-t4/adapter`, every `run.json` and `out/*/val/`.
Leave the `checkpoint-*` folders where they are.